# 00 — Build cities.csv
Fusiona tres fuentes para generar `data/processed/cities.csv`.

**Fuentes:**
- `cost_of_living_raw.csv` → índices de Kaggle
- `survey_results_public.csv` → salarios de Stack Overflow Survey
- `oecd_tax_raw.csv` → tipos impositivos de la OECD

**Produce:** `data/processed/cities.csv`

## 1. Imports y rutas

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(exist_ok=True)

# Tipo de cambio USD -> EUR (ajusta si es necesario)
USD_TO_EUR = 0.92

# Alquiler de referencia NYC en EUR/mes
# Usamos este valor para convertir rent_index a euros reales
# Fuente estimada: Numbeo 2023 (~3.500 USD = ~3.220 EUR)
NYC_RENT_EUR = 3220

## 2. Cargar Kaggle — cost of living
Columnas que nos interesan: `City`, `Cost of Living Index`, `Rent Index`, `Local Purchasing Power Index`

In [2]:
df_col = pd.read_csv(RAW / 'cost_of_living_raw.csv')

# Renombrar columnas al formato del proyecto
df_col = df_col.rename(columns={
    'City':                       'city_name',
    'Cost of Living Index':       'cost_of_living_index',
    'Rent Index':                 'rent_index',
    'Local Purchasing Power Index': 'purchasing_power_index',
})

# Quedarnos solo con las columnas necesarias
df_col = df_col[['city_name', 'cost_of_living_index', 'rent_index', 'purchasing_power_index']]

# Separar ciudad y país: el formato es "City, Country"
# Ejemplo: "Berlin, Germany" -> city="Berlin", country="Germany"
split = df_col['city_name'].str.rsplit(',', n=1, expand=True)
df_col['city_name'] = split[0].str.strip()
df_col['country']   = split[1].str.strip() if split.shape[1] > 1 else 'Unknown'

# Eliminar filas sin país
df_col = df_col.dropna(subset=['country'])

print(f'Ciudades cargadas: {len(df_col)}')
df_col.head(3)

Ciudades cargadas: 578


,city_name,cost_of_living_index,rent_index,purchasing_power_index,country
0,Hamilton,149.02,96.10,79.43,Bermuda
1,Zurich,131.24,69.26,129.79,Switzerland
2,Basel,130.93,49.38,111.53,Switzerland


## 3. Calcular average_rent en EUR

Formula:
```
average_rent = NYC_RENT_EUR * (rent_index / rent_index_NYC)
```

El `rent_index` de Nueva York en el dataset es ~100 (es la referencia base de Numbeo).
Si no aparece NYC, usamos 100 como valor de referencia directamente.

In [3]:
# Buscar el rent_index de Nueva York en el dataset
nyc_rows = df_col[df_col['city_name'].str.contains('New York', case=False, na=False)]

if len(nyc_rows) > 0:
    nyc_rent_index = nyc_rows.iloc[0]['rent_index']
    print(f'Rent index NYC encontrado: {nyc_rent_index}')
else:
    nyc_rent_index = 100  # valor base por defecto
    print('NYC no encontrada en dataset. Usando rent_index=100 como base.')

# Calcular alquiler medio estimado en EUR
df_col['average_rent'] = (NYC_RENT_EUR * (df_col['rent_index'] / nyc_rent_index)).round(0).astype(int)

print('\nEjemplo de alquileres calculados:')
df_col[['city_name', 'rent_index', 'average_rent']].head(5)

Rent index NYC encontrado: 100.0

Ejemplo de alquileres calculados:


,city_name,rent_index,average_rent
0,Hamilton,96.10,3094
1,Zurich,69.26,2230
2,Basel,49.38,1590
3,Zug,72.12,2322
4,Lugano,44.99,1449


## 4. Cargar Stack Overflow Survey — salarios por país

Del survey nos interesan:
- `Country` → país
- `ConvertedCompYearly` → salario anual en USD (ya convertido por SO)
- `DevType` → para filtrar solo developers

Calculamos la **mediana** por país (más robusta que la media ante outliers).

In [4]:
# El CSV del survey es grande (~100MB). Cargamos solo las columnas necesarias
df_so = pd.read_csv(
    RAW / 'survey_results_public.csv',
    usecols=['Country', 'ConvertedCompYearly', 'DevType', 'Employment']
)

print(f'Respuestas totales: {len(df_so)}')
print(f'Países únicos: {df_so["Country"].nunique()}')
df_so.head(2)

Respuestas totales: 49123
Países únicos: 177


,Employment,DevType,Country,ConvertedCompYearly
0,Employed,"Developer, mobile",Ukraine,61256.0
1,Employed,"Developer, back-end",Netherlands,104413.0


In [5]:
# Filtrar: solo empleados a tiempo completo con salario válido
df_so = df_so[
    (df_so['Employment'].str.contains('Employed', case=False, na=False)) &
    (df_so['ConvertedCompYearly'].notna()) &
    (df_so['ConvertedCompYearly'] > 5000) &    # eliminar salarios irreales
    (df_so['ConvertedCompYearly'] < 500000)    # eliminar outliers extremos
]

# Mediana de salario anual por país, convertida a EUR
df_salaries = (
    df_so.groupby('Country')['ConvertedCompYearly']
    .median()
    .reset_index()
    .rename(columns={'Country': 'country', 'ConvertedCompYearly': 'average_salary'})
)

# Convertir USD -> EUR
df_salaries['average_salary'] = (df_salaries['average_salary'] * USD_TO_EUR).round(0).astype(int)

print(f'Países con salario calculado: {len(df_salaries)}')
df_salaries.sort_values('average_salary', ascending=False).head(10)

Países con salario calculado: 155


,country,average_salary
99,North Korea,408893
101,Oman,358924
3,Andorra,184115
147,United States of America,138000
133,Switzerland,131185
62,Israel,129893
98,Nomadic,128081
83,Mauritania,111844
55,Iceland,108978
60,Ireland,106734


## 5. Cargar OECD — tax rate por país

El CSV de la OECD tiene un formato complejo. Filtramos:
- `TAX = TOP_TRATE` → tipo marginal superior
- `TIME_PERIOD` → el año más reciente disponible

In [6]:
df_oecd = pd.read_csv(RAW / 'oecd_tax_raw.csv')

print('Columnas OECD:', df_oecd.columns.tolist())
df_oecd.head(2)

Columnas OECD: ['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'COU', 'Unnamed: 5', 'TAX', 'Unnamed: 7', 'TIME_PERIOD', 'Unnamed: 9', 'OBS_VALUE', 'Unnamed: 11', 'OBS_STATUS', 'Unnamed: 13', 'UNIT_MEASURE', 'Unnamed: 15', 'UNIT_MULT', 'Unnamed: 17', 'BASE_PER', 'Unnamed: 19']


,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,COU,Unnamed: 5,TAX,Unnamed: 7,TIME_PERIOD,Unnamed: 9,OBS_VALUE,Unnamed: 11,OBS_STATUS,Unnamed: 13,UNIT_MEASURE,Unnamed: 15,UNIT_MULT,Unnamed: 17,BASE_PER,Unnamed: 19
0,DATAFLOW,OECD:DF_TABLE_I7(1.0),NaN,I,ISR,NaN,TOP_TRATE,NaN,2018,NaN,50.0,NaN,A,NaN,PC,NaN,0,NaN,NaN,NaN
1,DATAFLOW,OECD:DF_TABLE_I7(1.0),NaN,I,ISR,NaN,TOP_TRATE,NaN,2019,NaN,50.0,NaN,A,NaN,PC,NaN,0,NaN,NaN,NaN


In [7]:
# Filtrar tipo marginal superior (TOP_TRATE)
# Ajusta el nombre de columna si difiere al inspeccionarlo arriba
df_tax = df_oecd[df_oecd['TAX'] == 'TOP_TRATE'].copy()

# Quedarse con el año más reciente por país
df_tax = (
    df_tax.sort_values('TIME_PERIOD', ascending=False)
    .groupby('COU')
    .first()
    .reset_index()
    [['COU', 'OBS_VALUE']]
    .rename(columns={'OBS_VALUE': 'tax_rate'})
)

# La OECD usa códigos ISO3 (DEU, ESP, FRA...)
# Necesitamos mapearlos a nombres completos para cruzar con los otros datasets
# Usamos pycountry si está disponible, o un diccionario manual
try:
    import pycountry
    def iso3_to_name(code):
        try:
            return pycountry.countries.get(alpha_3=code).name
        except:
            return None
    df_tax['country'] = df_tax['COU'].apply(iso3_to_name)
except ImportError:
    # Diccionario manual con los países más comunes del dataset
    iso3_map = {
        'DEU': 'Germany',    'FRA': 'France',      'ESP': 'Spain',
        'ITA': 'Italy',      'NLD': 'Netherlands', 'BEL': 'Belgium',
        'CHE': 'Switzerland','AUT': 'Austria',     'SWE': 'Sweden',
        'NOR': 'Norway',     'DNK': 'Denmark',     'FIN': 'Finland',
        'GBR': 'United Kingdom', 'IRL': 'Ireland', 'PRT': 'Portugal',
        'POL': 'Poland',     'CZE': 'Czech Republic', 'HUN': 'Hungary',
        'USA': 'United States', 'CAN': 'Canada',   'AUS': 'Australia',
        'JPN': 'Japan',      'KOR': 'South Korea', 'ISR': 'Israel',
        'NZL': 'New Zealand','MEX': 'Mexico',      'TUR': 'Turkey',
        'GRC': 'Greece',     'SVK': 'Slovakia',    'LUX': 'Luxembourg',
        'EST': 'Estonia',    'LVA': 'Latvia',      'LTU': 'Lithuania',
        'SVN': 'Slovenia',   'HRV': 'Croatia',     'BGR': 'Bulgaria',
        'ROU': 'Romania',    'SGP': 'Singapore',   'ARE': 'United Arab Emirates',
    }
    df_tax['country'] = df_tax['COU'].map(iso3_map)

df_tax = df_tax.dropna(subset=['country'])[['country', 'tax_rate']]

print(f'Países con tax_rate: {len(df_tax)}')
df_tax.head(5)

Países con tax_rate: 34


,country,tax_rate
0,Australia,47.0000
1,Austria,55.0000
2,Belgium,52.8844
3,Canada,53.5296
4,Switzerland,41.5400


## 6. Añadir región manualmente

El dataset de Kaggle no incluye región. La añadimos con un diccionario.
Esto nos permite filtrar por Europa, Asia, etc. en la app.

In [8]:
region_map = {
    # Europa occidental
    'Germany':'Europe', 'France':'Europe', 'Spain':'Europe', 'Italy':'Europe',
    'Netherlands':'Europe', 'Belgium':'Europe', 'Switzerland':'Europe',
    'Austria':'Europe', 'Portugal':'Europe', 'Ireland':'Europe',
    'United Kingdom':'Europe', 'Luxembourg':'Europe', 'Denmark':'Europe',
    'Sweden':'Europe', 'Norway':'Europe', 'Finland':'Europe',
    # Europa del este
    'Poland':'Europe', 'Czech Republic':'Europe', 'Hungary':'Europe',
    'Romania':'Europe', 'Bulgaria':'Europe', 'Croatia':'Europe',
    'Slovakia':'Europe', 'Slovenia':'Europe', 'Estonia':'Europe',
    'Latvia':'Europe', 'Lithuania':'Europe', 'Greece':'Europe',
    # América
    'United States':'Americas', 'Canada':'Americas', 'Mexico':'Americas',
    'Brazil':'Americas', 'Argentina':'Americas', 'Colombia':'Americas',
    # Asia
    'Japan':'Asia', 'South Korea':'Asia', 'Singapore':'Asia',
    'China':'Asia', 'India':'Asia', 'Taiwan':'Asia', 'Hong Kong':'Asia',
    # Oriente Medio
    'United Arab Emirates':'Middle East', 'Israel':'Middle East',
    'Qatar':'Middle East', 'Saudi Arabia':'Middle East',
    # Oceanía
    'Australia':'Oceania', 'New Zealand':'Oceania',
    # África
    'South Africa':'Africa',
}

print(f'Países en el mapa de regiones: {len(region_map)}')

Países en el mapa de regiones: 48


## 7. Fusionar los tres datasets

El campo de unión es `country`. Usamos `left join` desde el dataset de ciudades
para no perder ciudades aunque falten datos de salario o tax en algún país.

In [9]:
# Base: ciudades con índices
df = df_col.copy()

# Unir salarios por país
df = df.merge(df_salaries, on='country', how='left')

# Unir tax_rate por país
df = df.merge(df_tax, on='country', how='left')

# Añadir región
df['region'] = df['country'].map(region_map).fillna('Other')

# job_market_score: aproximación simple basada en purchasing_power_index
# (en ausencia de una fuente específica, lo normalizamos al rango 0-100)
ppi_min = df['purchasing_power_index'].min()
ppi_max = df['purchasing_power_index'].max()
df['job_market_score'] = (
    (df['purchasing_power_index'] - ppi_min) / (ppi_max - ppi_min) * 100
).round(1)

# Rellenar tax_rate faltante con la mediana global
median_tax = df['tax_rate'].median()
df['tax_rate'] = df['tax_rate'].fillna(median_tax).round(1)

# Rellenar salarios faltantes con la mediana global
median_salary = df['average_salary'].median()
df['average_salary'] = df['average_salary'].fillna(median_salary).round(0).astype(int)

print(f'Ciudades totales: {len(df)}')
print(f'Ciudades con todos los datos: {df.dropna().shape[0]}')
df.head(5)

Ciudades totales: 578
Ciudades con todos los datos: 578


,city_name,cost_of_living_index,rent_index,purchasing_power_index,country,average_rent,average_salary,tax_rate,region,job_market_score
0,Hamilton,149.02,96.10,79.43,Bermuda,3094,52107,45.0,Other,45.4
1,Zurich,131.24,69.26,129.79,Switzerland,2230,131185,41.5,Europe,74.8
2,Basel,130.93,49.38,111.53,Switzerland,1590,131185,41.5,Europe,64.1
3,Zug,128.13,72.12,143.40,Switzerland,2322,131185,41.5,Europe,82.7
4,Lugano,123.99,44.99,111.96,Switzerland,1449,131185,41.5,Europe,64.4


## 8. Ordenar columnas y guardar

El CSV final tiene exactamente las columnas que espera `src/data_loader.py`.

In [10]:
# Columnas en el orden exacto que espera data_loader.py
columns_final = [
    'city_name',
    'country',
    'region',
    'average_salary',
    'average_rent',
    'tax_rate',
    'cost_of_living_index',
    'purchasing_power_index',
    'quality_of_life_index',   # puede ser NaN si no está en fuente
    'job_market_score',
]

# quality_of_life_index no viene de ninguna fuente todavía
# Lo aproximamos con una combinación de purchasing_power y tax_rate
if 'quality_of_life_index' not in df.columns:
    df['quality_of_life_index'] = (
        df['purchasing_power_index'] * 0.6 +
        (100 - df['tax_rate']) * 0.4
    ).round(1)

df_final = df[columns_final].copy()

# Guardar en data/processed/
output_path = PROCESSED / 'cities.csv'
df_final.to_csv(output_path, index=False)

print(f'Guardado: {output_path}')
print(f'Shape: {df_final.shape}')
df_final.head(5)

Guardado: ..\data\processed\cities.csv
Shape: (578, 10)


,city_name,country,region,average_salary,average_rent,tax_rate,cost_of_living_index,purchasing_power_index,quality_of_life_index,job_market_score
0,Hamilton,Bermuda,Other,52107,3094,45.0,149.02,79.43,69.7,45.4
1,Zurich,Switzerland,Europe,131185,2230,41.5,131.24,129.79,101.3,74.8
2,Basel,Switzerland,Europe,131185,1590,41.5,130.93,111.53,90.3,64.1
3,Zug,Switzerland,Europe,131185,2322,41.5,128.13,143.40,109.4,82.7
4,Lugano,Switzerland,Europe,131185,1449,41.5,123.99,111.96,90.6,64.4


## 9. Verificación rápida

Comprobamos que el archivo generado tiene sentido antes de usarlo en la app.

In [11]:
df_check = pd.read_csv(PROCESSED / 'cities.csv')

print('=== Resumen ===')
print(f'Ciudades: {len(df_check)}')
print(f'Paises:   {df_check["country"].nunique()}')
print(f'Regiones: {df_check["region"].value_counts().to_dict()}')
print()
print('=== Valores nulos ===')
print(df_check.isnull().sum())
print()
print('=== Top 5 ciudades por purchasing_power_index ===')
print(df_check.nlargest(5, 'purchasing_power_index')[['city_name', 'country', 'average_salary', 'average_rent', 'purchasing_power_index']])

=== Resumen ===
Ciudades: 578
Paises:   127
Regiones: {'Europe': 210, 'Americas': 142, 'Other': 131, 'Asia': 63, 'Oceania': 15, 'Middle East': 13, 'Africa': 4}

=== Valores nulos ===
city_name                 0
country                   0
region                    0
average_salary            0
average_rent              0
tax_rate                  0
cost_of_living_index      0
purchasing_power_index    0
quality_of_life_index     0
job_market_score          0
dtype: int64

=== Top 5 ciudades por purchasing_power_index ===
         city_name        country  average_salary  average_rent  \
276    Houston, TX  United States           52107          1397   
233     Dallas, TX  United States           52107          1615   
190  Ann Arbor, MI  United States           52107          1545   
250     Austin, TX  United States           52107          1857   
130   San Jose, CA  United States           52107          2650   

     purchasing_power_index  
276                  172.98  
233       

Cargar Coordenadas por Ciudad

In [5]:
import pandas as pd
from pathlib import Path

RAW = Path('../data/raw')

# Ver exactamente qué columnas tiene el archivo
df_test = pd.read_csv(RAW / 'worldcitiespop.csv', nrows=5)
print("Columnas:", df_test.columns.tolist())
print()
print(df_test)

Columnas: ['Country', 'City', 'AccentCity', 'Region', 'Population', 'Latitude', 'Longitude']

  Country        City  AccentCity  Region  Population   Latitude  Longitude
0      ad       aixas       Aixàs       6         NaN  42.483333   1.466667
1      ad  aixirivali  Aixirivali       6         NaN  42.466667   1.500000
2      ad  aixirivall  Aixirivall       6         NaN  42.466667   1.500000
3      ad   aixirvall   Aixirvall       6         NaN  42.466667   1.500000
4      ad    aixovall    Aixovall       6         NaN  42.466667   1.483333


In [9]:
import pandas as pd
from pathlib import Path

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')

# Mapa ISO2 -> nombre completo
iso2_map = {
    'de':'Germany','fr':'France','es':'Spain','it':'Italy','nl':'Netherlands',
    'be':'Belgium','ch':'Switzerland','at':'Austria','pt':'Portugal','ie':'Ireland',
    'gb':'United Kingdom','lu':'Luxembourg','dk':'Denmark','se':'Sweden','no':'Norway',
    'fi':'Finland','pl':'Poland','cz':'Czech Republic','hu':'Hungary','ro':'Romania',
    'bg':'Bulgaria','hr':'Croatia','sk':'Slovakia','si':'Slovenia','ee':'Estonia',
    'lv':'Latvia','lt':'Lithuania','gr':'Greece','us':'United States','ca':'Canada',
    'au':'Australia','jp':'Japan','kr':'South Korea','sg':'Singapore','in':'India',
    'br':'Brazil','mx':'Mexico','ar':'Argentina','cn':'China','ae':'United Arab Emirates',
    'il':'Israel','nz':'New Zealand','za':'South Africa','lb':'Lebanon','bm':'Bermuda',
    'tr':'Turkey','mx':'Mexico','ru':'Russia','ua':'Ukraine','rs':'Serbia',
}

# Cargar worldcitiespop con AccentCity
df_coords = pd.read_csv(
    RAW / 'worldcitiespop.csv',
    usecols=['AccentCity', 'Country', 'Latitude', 'Longitude'],
    low_memory=False
)

# Mapear código ISO2 a nombre completo
df_coords['country'] = df_coords['Country'].str.lower().map(iso2_map)
df_coords = df_coords.dropna(subset=['country'])
df_coords = df_coords.rename(columns={'AccentCity': 'city_name', 'Latitude': 'lat', 'Longitude': 'lon'})
df_coords = df_coords[['city_name', 'country', 'lat', 'lon']]
df_coords = df_coords.drop_duplicates(subset=['city_name', 'country'])

# Cargar cities_processed
df_final = pd.read_csv(PROCESSED / 'cities_processed.csv')

df_final = df_final.drop(columns=['lat', 'lon'], errors='ignore')

# Diagnóstico antes del merge
print("=== Ejemplo ciudades en worldcitiespop (Germany) ===")
print(df_coords[df_coords['country'] == 'Germany']['city_name'].head(10).tolist())

print("\n=== Ciudades en cities_processed ===")
print(df_final['city_name'].tolist())

# Merge
df_merged = df_final.merge(
    df_coords,
    on=['city_name', 'country'],
    how='left'
)

encontradas = df_merged['lat'].notna().sum()
print(f"\nCiudades con coordenadas: {encontradas} / {len(df_merged)}")

if encontradas == 0:
    print("\nNinguna coincidencia — revisando nombres:")
    for city, country in zip(df_final['city_name'].head(5), df_final['country'].head(5)):
        match = df_coords[
            (df_coords['country'] == country) &
            (df_coords['city_name'].str.lower() == city.lower())
        ]
        print(f"  {city}, {country} -> {'ENCONTRADA' if len(match) > 0 else 'NO encontrada'}")

# Guardar solo si hay coincidencias
if encontradas > 0:
    df_merged.to_csv(PROCESSED / 'cities_processed.csv', index=False)
    print("\nGuardado correctamente.")

=== Ejemplo ciudades en worldcitiespop (Germany) ===
['Aabauerschaft', 'Aach', 'Aachen', 'Aachen-Rothe Erde', 'Aach-Linz', 'Aaldering', 'Aalderink', 'Aalen', 'Aalkorb', 'Aasbüttel']

=== Ciudades en cities_processed ===
['Hamilton', 'Zurich', 'Basel', 'Zug', 'Lugano', 'Lausanne', 'Beirut', 'Bern', 'Geneva', 'Stavanger', 'Honolulu, HI', 'Oslo', 'Bergen', 'New York, NY', 'Trondheim', 'Tromso', 'Reykjavik', 'Saint Helier', 'Santa Barbara, CA', 'Tel Aviv-Yafo', 'Berkeley, CA', 'San Francisco, CA', 'Oakland, CA', 'Anchorage, AK', 'Santa Clara, CA', 'Petah Tikva', 'Beersheba', 'Seattle, WA', 'Copenhagen', 'Arhus', 'Haifa', 'Jerusalem', 'Nassau', 'London', 'Tokyo', 'Boston, MA', 'Nice', 'Paris', 'Queens, NY', 'Singapore', 'Washington, DC', 'Sydney', 'Luxembourg', 'Espoo', 'Aalborg', 'Pittsburgh, PA', 'Darwin', 'Amsterdam', 'Jersey City, NJ', 'Hong Kong', 'Breda', 'Nanaimo, BC', 'Auckland', 'Haarlem', 'Utrecht', 'Brighton', 'Brisbane', 'Philadelphia, PA', 'Los Angeles, CA', 'Minneapolis, MN', 

In [7]:
import pandas as pd
from pathlib import Path

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')

# Ver muestra de worldcitiespop
df_coords = pd.read_csv(RAW / 'worldcitiespop.csv', usecols=['AccentCity', 'Country'], low_memory=False)
print("=== worldcitiespop — muestra ===")
print(df_coords[df_coords['Country'] == 'de'].head(10))

# Ver muestra de cities_processed
df_cities = pd.read_csv(PROCESSED / 'cities_processed.csv')
print("\n=== cities_processed — muestra ===")
print(df_cities[['city_name', 'country']].head(10))

=== worldcitiespop — muestra ===
       Country         AccentCity
722412      de      Aabauerschaft
722413      de               Aach
722414      de               Aach
722415      de               Aach
722416      de             Aachen
722417      de  Aachen-Rothe Erde
722418      de          Aach-Linz
722419      de          Aaldering
722420      de          Aalderink
722421      de              Aalen

=== cities_processed — muestra ===
   city_name      country
0   Hamilton      Bermuda
1     Zurich  Switzerland
2      Basel  Switzerland
3        Zug  Switzerland
4     Lugano  Switzerland
5   Lausanne  Switzerland
6     Beirut      Lebanon
7       Bern  Switzerland
8     Geneva  Switzerland
9  Stavanger       Norway
